# 🏢 Leads BI Dashboard
### Fuentes: DENUE (INEGI) + CompraNet
---

---
## 🔄 Pipeline de Extracción Integrado
> Corre los tres extractores desde el notebook sin salir a la terminal

In [24]:
# ── CONFIGURACIÓN GLOBAL ─────────────────────────────────────
import subprocess, sys, os, glob, io, time, warnings, json
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── Rutas ────────────────────────────────────────────────────
RUTA_TRABAJO = '/home/carlos_cognitia/Sync/Desarrollos/Leads'   # carpeta donde se guardan todos los CSVs

# ── DENUE ────────────────────────────────────────────────────
DENUE_TOKEN    = 'DENUE_ENTIDADES = ['09', '15']   # 09=CDMX | 15=EdoMex | 19=NL | 14=Jalisco
DENUE_CONDICION = 'todos'
DENUE_PAGINA    = 1000

# ── CompraNet ────────────────────────────────────────────────
CNET_MONTO_MINIMO  = 50_000
CNET_SOLO_MEXICANAS = True
CNET_ARCHIVOS = {
    '2025_contratos':   'https://upcp-compranet.buengobierno.gob.mx/cnetassets/datos_abiertos_contratos_expedientes/Contratos_CompraNet2025.csv',
    '2025_expedientes': 'https://upcp-compranet.buengobierno.gob.mx/cnetassets/datos_abiertos_contratos_expedientes/Expedientes_PICompraNet2025.csv',
    '2024_contratos':   'https://comprasmx.buengobierno.gob.mx/cnetassets/datos_abiertos_contratos_expedientes/Contratos_CompraNet2024.csv',
    '2024_expedientes': 'https://comprasmx.buengobierno.gob.mx/cnetassets/datos_abiertos_contratos_expedientes/Expedientes_PICompraNet2024.csv',
}

# ── DataMéxico ───────────────────────────────────────────────
DATAMX_BASE  = 'https://api.datamexico.org/tesseract/data.jsonrecords'
DATAMX_ANIOS = [2022, 2023, 2024]

HEADERS_JSON = {
    'User-Agent': 'Mozilla/5.0 Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json',
}
HEADERS_CSV = {
    'User-Agent': 'Mozilla/5.0 Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/csv,*/*',
}

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print('✅ Configuración cargada')
print(f'   DENUE entidades: {DENUE_ENTIDADES}')
print(f'   CompraNet archivos: {list(CNET_ARCHIVOS.keys())}')
print(f'   DataMéxico años: {DATAMX_ANIOS}')


✅ Configuración cargada
   DENUE entidades: ['09', '15']
   CompraNet archivos: ['2025_contratos', '2025_expedientes', '2024_contratos', '2024_expedientes']
   DataMéxico años: [2022, 2023, 2024]


### 1️⃣-A Extractor DENUE (INEGI)

In [25]:
from tqdm.auto import tqdm
# ── DENUE: funciones de extracción ───────────────────────────
DENUE_API = 'https://www.inegi.org.mx/app/api/denue/v1/consulta'
ENTIDADES_MX = {
    '01':'Aguascalientes','02':'Baja California','03':'Baja California Sur',
    '04':'Campeche','05':'Coahuila','06':'Colima','07':'Chiapas','08':'Chihuahua',
    '09':'Ciudad de México','10':'Durango','11':'Guanajuato','12':'Guerrero',
    '13':'Hidalgo','14':'Jalisco','15':'Estado de México','16':'Michoacán',
    '17':'Morelos','18':'Nayarit','19':'Nuevo León','20':'Oaxaca',
    '21':'Puebla','22':'Querétaro','23':'Quintana Roo','24':'San Luis Potosí',
    '25':'Sinaloa','26':'Sonora','27':'Tabasco','28':'Tamaulipas',
    '29':'Tlaxcala','30':'Veracruz','31':'Yucatán','32':'Zacatecas',
}

def denue_get(url):
    for i in range(3):
        try:
            r = requests.get(url, headers=HEADERS_JSON, timeout=60)
            if r.status_code == 200: return r.json()
        except Exception as e:
            if i < 2: time.sleep(2**i)
    return None

def denue_parsear(item, entidad):
    return {
        'id': item.get('Id',''), 'clee': item.get('CLEE',''),
        'nombre': item.get('Nombre','').strip(),
        'razon_social': item.get('Razon_social','').strip(),
        'actividad_economica': item.get('Clase_actividad','').strip(),
        'id_sector': item.get('SECTOR_ACTIVIDAD_ID',''),
        'estrato': item.get('Estrato',''),
        'telefono': item.get('Telefono','').strip(),
        'email': item.get('Correo_e','').strip().lower(),
        'sitio_web': item.get('Sitio_internet','').strip(),
        'calle': item.get('Calle','').strip(),
        'colonia': item.get('Colonia','').strip(),
        'cp': item.get('CP',''),
        'ubicacion': item.get('Ubicacion','').strip(),
        'latitud': item.get('Latitud',''), 'longitud': item.get('Longitud',''),
        'fecha_alta': item.get('Fecha_Alta',''),
        'entidad_clave': entidad,
        'entidad_nombre': ENTIDADES_MX.get(entidad, entidad),
        'fuente': 'DENUE_INEGI',
        'fecha_extraccion': datetime.now().strftime('%Y-%m-%d'),
    }

def denue_extraer_entidad(entidad, condicion='todos', pagina=1000):
    nombre = ENTIDADES_MX.get(entidad, entidad)
    print(f'📍 Extrayendo: {nombre} ({entidad})')
    registros = {}
    ini = 1
    sin_datos = 0
    primera = denue_get(f'{DENUE_API}/BuscarEntidad/{condicion}/{entidad}/{ini}/{ini+pagina-1}/{DENUE_TOKEN}')
    if not isinstance(primera, list) or not primera:
        print(f'  ⚠️  Sin datos'); return pd.DataFrame()
    for item in primera:
        if (id_ := item.get('Id','')): registros[id_] = denue_parsear(item, entidad)
    ini += pagina
    with tqdm(desc=f'  {nombre[:18]}', unit='reg', unit_scale=True) as bar:
        bar.update(len(registros))
        while True:
            data = denue_get(f'{DENUE_API}/BuscarEntidad/{condicion}/{entidad}/{ini}/{ini+pagina-1}/{DENUE_TOKEN}')
            if not isinstance(data, list) or not data:
                sin_datos += 1
                if sin_datos >= 2: break
                ini += pagina; time.sleep(1); continue
            sin_datos = 0
            for item in data:
                if (id_ := item.get('Id','')) and id_ not in registros:
                    registros[id_] = denue_parsear(item, entidad)
            bar.update(len(data))
            bar.set_postfix({'únicos': len(registros)})
            if len(data) < pagina: break
            ini += pagina; time.sleep(0.5)
    df = pd.DataFrame(list(registros.values()))
    print(f'  ✅ {len(df):,} registros únicos')
    return df

print('✅ Funciones DENUE listas')


✅ Funciones DENUE listas


In [3]:
# ── DENUE: CORRER EXTRACCIÓN ──────────────────────────────────
# Verifica conexión
test = denue_get(f'{DENUE_API}/BuscarEntidad/todos/09/1/3/{DENUE_TOKEN}')
if not isinstance(test, list) or not test:
    print('❌ Sin conexión DENUE — verifica el token')
else:
    print(f'✅ DENUE OK — {test[0].get("Nombre","?")} | {test[0].get("Ubicacion","?")}')
    
    dfs_denue = []
    for ent in DENUE_ENTIDADES:
        df_e = denue_extraer_entidad(ent, DENUE_CONDICION, DENUE_PAGINA)
        if not df_e.empty: dfs_denue.append(df_e)
        time.sleep(1)

    if dfs_denue:
        df_denue = pd.concat(dfs_denue, ignore_index=True)
        df_denue = df_denue.drop_duplicates(subset=['id'])
        df_denue = df_denue[df_denue['nombre'].str.strip() != '']
        df_denue['telefono'] = df_denue['telefono'].str.replace(r'[^\d\s\-\+\(\)]','',regex=True).str.strip()
        archivo_denue = os.path.join(RUTA_TRABAJO, f'leads_denue_{TIMESTAMP}.csv')
        df_denue.to_csv(archivo_denue, index=False, encoding='utf-8-sig')
        print(f'\n📈 DENUE: {len(df_denue):,} leads')
        print(f'   Con teléfono:  {(df_denue["telefono"].str.len()>6).sum():,}')
        print(f'   Con email:     {(df_denue["email"].str.len()>5).sum():,}')
        print(f'   Con sitio web: {(df_denue["sitio_web"].str.len()>5).sum():,}')
        print(f'💾 {archivo_denue}')
    else:
        print('⚠️  Sin datos DENUE')
        df_denue = pd.DataFrame()


✅ DENUE OK —  ARELLANO AYALA ABOGADOS | MIGUEL HIDALGO, Miguel Hidalgo, CIUDAD DE MÉXICO
📍 Extrayendo: Ciudad de México (09)


  Ciudad de México: 461kreg [19:38, 391reg/s, únicos=460762] 


  ✅ 460,762 registros únicos
📍 Extrayendo: Estado de México (15)


  Estado de México: 817kreg [45:23, 300reg/s, únicos=816716] 


  ✅ 816,716 registros únicos

📈 DENUE: 1,277,478 leads
   Con teléfono:  385,002
   Con email:     170,619
   Con sitio web: 74,437
💾 ./leads_denue_20260415_172136.csv


### 1️⃣-B Cargar data del DENUE (INEGI)

In [43]:
# Buscar archivos tipo leads_denue_*.csv
pattern = os.path.join(RUTA_TRABAJO, "leads_denue_*.csv")

archivos = glob.glob(pattern)

if not archivos:
    raise FileNotFoundError("No se encontraron archivos leads_denue")

# Obtener el más reciente por fecha de modificación
archivo_reciente = max(archivos, key=os.path.getmtime)

print("📂 Archivo más reciente:", archivo_reciente)

# Cargar
df_denue = pd.read_csv(archivo_reciente, encoding='utf-8-sig')

📂 Archivo más reciente: /home/carlos_cognitia/Sync/Desarrollos/Leads/leads_denue_20260413_183347.csv


In [44]:
df_denue


,id,clee,nombre,razon_social,actividad_economica,id_clase,id_sector,id_subsector,estrato,telefono,...,longitud,tipo_establecimiento,corredor,ageb,fecha_alta,area_geo,entidad_clave,entidad_nombre,fuente,fecha_extraccion
0,762690,09016541110003013001000000U0,ARELLANO AYALA ABOGADOS,NaN,Bufetes jurídicos,NaN,NaN,NaN,0 a 5 personas,NaN,...,-99.204354,Fijo,NaN,NaN,NaN,NaN,9,Ciudad de México,DENUE_INEGI,2026-04-13
1,11647314,09011722513005291000086441M7,BARBACOA MALENO,NaN,Restaurantes con servicio de preparación de an...,NaN,NaN,NaN,0 a 5 personas,NaN,...,-99.046591,Fijo,MERCADO MIGUEL HIDALGO,NaN,NaN,NaN,9,Ciudad de México,DENUE_INEGI,2026-04-13
2,6730131,09002468111000232001000000U6,BMW LINDAVISTA,LINDAVISTA MOTORS SA DE CV,Comercio al por menor de automóviles y camione...,NaN,NaN,NaN,101 a 250 personas,NaN,...,-99.156022,Fijo,INDUSTRIAL VALLEJO,NaN,NaN,NaN,9,Ciudad de México,DENUE_INEGI,2026-04-13
3,6324838,09011431192000031001007670S7,CEDIS IZTAPALAPA DTS,COMERCIALIZADORA PEPSICO MEXICO S DE RL DE CV,Comercio al por mayor de botanas y frituras,NaN,NaN,NaN,31 a 50 personas,NaN,...,-99.094808,Fijo,NaN,NaN,NaN,NaN,9,Ciudad de México,DENUE_INEGI,2026-04-13
4,6319760,09017465915000021000003945S8,FONDO NACIONAL PARA EL FOMENTO A LAS ARTESANIA...,FONDO NACIONAL PARA EL FOMENTO DE LAS ARTESANIAS,Comercio al por menor en tiendas de artesanías,NaN,NaN,NaN,11 a 30 personas,NaN,...,-99.080005,Fijo,NaN,NaN,NaN,NaN,9,Ciudad de México,DENUE_INEGI,2026-04-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1277473,10265316,15104713120001881000000000U9,ZYL GAMES,NaN,Casas de juegos electrónicos,NaN,NaN,NaN,0 a 5 personas,NaN,...,-99.185028,Fijo,NaN,NaN,NaN,NaN,15,Estado de México,DENUE_INEGI,2026-04-13
1277474,9451346,15104466114000653000000000U5,ZYLKU IMPORTACIONES,ZYLKU IMPORTACIONES SA DE CV,"Comercio al por menor de cristalería, loza y u...",NaN,NaN,NaN,0 a 5 personas,NaN,...,-99.196816,Fijo,NaN,NaN,NaN,NaN,15,Estado de México,DENUE_INEGI,2026-04-13
1277475,2533061,15076463310006681000000000U7,ZZENIA,NaN,Comercio al por menor de calzado,NaN,NaN,NaN,0 a 5 personas,7221812004.0,...,-99.532420,Fijo,PROCASMAC PLAZA NARANJA,NaN,NaN,NaN,15,Estado de México,DENUE_INEGI,2026-04-13
1277476,11587588,15076463310014241000000000S0,ZZENIA,NaN,Comercio al por menor de calzado,NaN,NaN,NaN,0 a 5 personas,7221812004.0,...,-99.529026,Fijo,PLAZA NARANJA 2000,NaN,NaN,NaN,15,Estado de México,DENUE_INEGI,2026-04-13


### 1️⃣-B Extractor CompraNet

In [45]:
from tqdm.auto import tqdm
# ── COMPRANET: funciones ─────────────────────────────────────
CNET_BUSQUEDA = {
    'rfc':               ['rfc'],
    'razon_social':      ['proveedor','contratista','razon social'],
    'monto_contrato':    ['importe','monto','precio'],
    'estatus':           ['estatus contrato','estatus','status'],
    'tipo_contratacion': ['tipo de contratacion','tipo contratacion'],
    'fecha_inicio':      ['fecha inicio','fecha de inicio'],
    'dependencia':       ['descripcion ramo','descripción ramo','ramo'],
    'unidad_compradora': ['nombre de la uc','nombre uc'],
    'tamanio_empresa':   ['estratificacion','estratificación','mipyme'],
    'nacionalidad':      ['nacionalidad'],
    'moneda':            ['moneda'],
}

def cnet_detectar_columnas(df):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    mapeo = {}
    for campo, palabras in CNET_BUSQUEDA.items():
        for col_lower, col_real in cols_lower.items():
            for palabra in palabras:
                if palabra in col_lower:
                    mapeo[campo] = col_real; break
            if campo in mapeo: break
    return mapeo

def cnet_descargar(nombre, url):
    print(f'\n📥 {nombre}')
    try:
        with requests.get(url, headers=HEADERS_CSV, stream=True, timeout=180) as r:
            if r.status_code != 200:
                print(f'  ❌ HTTP {r.status_code}'); return None
            total = int(r.headers.get('content-length', 0))
            print(f'  📦 {total/1024/1024:.1f} MB')
            chunks = []
            with tqdm(total=total, unit='B', unit_scale=True, desc='  ⬇', leave=False) as bar:
                for chunk in r.iter_content(chunk_size=512*1024):
                    chunks.append(chunk); bar.update(len(chunk))
            data = b''.join(chunks)
            print(f'  ✅ {len(data)/1024/1024:.1f} MB')
            return data
    except Exception as e:
        print(f'  ❌ {e}'); return None

def cnet_leer(data):
    for enc in ['latin-1','utf-8','utf-8-sig','cp1252']:
        try:
            df = pd.read_csv(io.BytesIO(data), encoding=enc, low_memory=False, on_bad_lines='skip')
            print(f'  📊 {len(df):,} filas | {enc}')
            return df
        except: continue
    return None

def cnet_procesar(df, nombre):
    mapeo = cnet_detectar_columnas(df)
    filas = [{campo: row.get(col,'') for campo, col in mapeo.items()} for _, row in df.iterrows()]
    df_c = pd.DataFrame(filas)
    if 'monto_contrato' in df_c.columns:
        df_c['monto_contrato'] = pd.to_numeric(
            df_c['monto_contrato'].astype(str).str.replace(',','').str.replace('$','').str.strip(),
            errors='coerce').fillna(0)
    for col in ['rfc','razon_social','estatus','nacionalidad','tamanio_empresa']:
        if col in df_c.columns:
            df_c[col] = df_c[col].astype(str).str.strip().str.upper()
    df_c['archivo_origen'] = nombre
    df_c['fuente'] = 'COMPRANET'
    df_c['fecha_extraccion'] = datetime.now().strftime('%Y-%m-%d')
    print(f'  📈 Antes filtrar: {len(df_c):,}')
    if 'estatus' in df_c.columns:
        df_c = df_c[df_c['estatus'].str.contains('FORMALIZADO', na=False)]
        print(f'  🔍 Formalizados: {len(df_c):,}')
    if 'monto_contrato' in df_c.columns:
        df_c = df_c[df_c['monto_contrato'] >= CNET_MONTO_MINIMO]
        print(f'  🔍 Monto ≥ ${CNET_MONTO_MINIMO:,}: {len(df_c):,}')
    if CNET_SOLO_MEXICANAS and 'nacionalidad' in df_c.columns:
        df_c = df_c[df_c['nacionalidad'].str.contains('MEXICAN|MX|NACION', na=False, regex=True)]
        print(f'  🔍 Mexicanas: {len(df_c):,}')
    if 'rfc' in df_c.columns:
        df_c = df_c[df_c['rfc'].astype(str).str.len() >= 12]
        print(f'  🔍 RFC válido: {len(df_c):,}')
    return df_c

def cnet_agregar(df):
    if 'rfc' not in df.columns or df.empty: return df
    agg = {'razon_social':('razon_social','first'),'num_contratos':('rfc','count')}
    if 'monto_contrato' in df.columns:
        agg['monto_total_MXN'] = ('monto_contrato','sum')
        agg['monto_max']       = ('monto_contrato','max')
    if 'tamanio_empresa' in df.columns: agg['tamanio_empresa'] = ('tamanio_empresa','first')
    if 'dependencia'     in df.columns: agg['dependencias'] = ('dependencia', lambda x: ' | '.join(x.dropna().astype(str).unique()[:3]))
    if 'fecha_inicio'    in df.columns: agg['ultimo_contrato'] = ('fecha_inicio','max')
    result = df.groupby('rfc').agg(**agg).reset_index()
    if 'monto_total_MXN' in result.columns:
        result['score_lead'] = ((result['num_contratos']*10) + (result['monto_total_MXN']/100_000).clip(upper=100)).round(1)
        result = result.sort_values('score_lead', ascending=False)
    return result

print('✅ Funciones CompraNet listas')


✅ Funciones CompraNet listas


In [32]:
# ── COMPRANET: CORRER EXTRACCIÓN ─────────────────────────────
dfs_cnet = []
for nombre, url in CNET_ARCHIVOS.items():
    raw = cnet_descargar(nombre, url)
    if raw is None: continue
    df_raw = cnet_leer(raw)
    if df_raw is None: continue
    df_proc = cnet_procesar(df_raw, nombre)
    if not df_proc.empty: dfs_cnet.append(df_proc)

if dfs_cnet:
    df_cnet = pd.concat(dfs_cnet, ignore_index=True)
    df_emp  = cnet_agregar(df_cnet)
    archivo_cnet = os.path.join(RUTA_TRABAJO, f'leads_compranet_{TIMESTAMP}_contratos.csv')
    archivo_emp  = os.path.join(RUTA_TRABAJO, f'leads_compranet_{TIMESTAMP}_empresas.csv')
    df_cnet.to_csv(archivo_cnet, index=False, encoding='utf-8-sig')
    df_emp.to_csv(archivo_emp, index=False, encoding='utf-8-sig')
    print(f'\n📈 CompraNet: {len(df_cnet):,} contratos | {len(df_emp):,} empresas únicas')
    if 'monto_contrato' in df_cnet.columns:
        print(f'   Monto total: ${df_cnet["monto_contrato"].sum():,.0f} MXN')
    print(f'💾 {archivo_cnet}')
    print(f'💾 {archivo_emp}')
    if 'score_lead' in df_emp.columns:
        print(f'\n🏆 Top 5 leads CompraNet:')
        cols = [c for c in ['razon_social','rfc','num_contratos','monto_total_MXN','score_lead'] if c in df_emp.columns]
        print(df_emp[cols].head(5).to_string(index=False))
else:
    print('⚠️  Sin datos CompraNet')
    df_cnet = pd.DataFrame()
    df_emp  = pd.DataFrame()



📥 2025_contratos
  📦 105.0 MB


  ✅ 105.0 MB
  📊 91,850 filas | latin-1
  📈 Antes filtrar: 91,850
  🔍 Formalizados: 0
  🔍 Monto ≥ $50,000: 0
  🔍 Mexicanas: 0
  🔍 RFC válido: 0

📥 2025_expedientes
  📦 69.4 MB


  ✅ 69.4 MB
  📊 65,486 filas | latin-1
  📈 Antes filtrar: 65,486

📥 2024_contratos
  📦 160.4 MB


  ✅ 160.4 MB
  📊 143,542 filas | latin-1
  📈 Antes filtrar: 143,542
  🔍 Formalizados: 0
  🔍 Monto ≥ $50,000: 0
  🔍 Mexicanas: 0
  🔍 RFC válido: 0

📥 2024_expedientes
  📦 105.3 MB


  ✅ 105.3 MB
  📊 100,917 filas | latin-1
  📈 Antes filtrar: 100,917

📈 CompraNet: 166,403 contratos | 166,403 empresas únicas
💾 /home/carlos_cognitia/Sync/Desarrollos/Leads/leads_compranet_20260416_135503_contratos.csv
💾 /home/carlos_cognitia/Sync/Desarrollos/Leads/leads_compranet_20260416_135503_empresas.csv


### 1️⃣-C Extractor DataMéxico API

In [46]:
DATAMX_BASE = "https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords"
print(DATAMX_BASE)

def datamx_get(params):
    for i in range(3):
        try:
            print("URL base usada:", DATAMX_BASE)
            r = requests.get(
                DATAMX_BASE,
                params=params,
                headers=HEADERS_JSON,
                timeout=60
            )

            if r.status_code == 200:
                payload = r.json()
                if isinstance(payload, dict):
                    return payload.get("data", [])
                print("  ⚠️ Respuesta 200 pero formato inesperado")
                return []

            print(f"  ⚠️ HTTP {r.status_code} :: {r.text[:300]}")

        except Exception as e:
            print(f"  ⚠️ intento {i+1} falló: {e}")
            if i < 2:
                time.sleep(2**i)
    return []



def datamx_perfil_estados(df_comercio):
    if df_comercio.empty:
        return pd.DataFrame()

    pivot = (
        df_comercio.groupby(["estado", "tipo_operacion"])["valor_usd"]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )
    pivot.columns.name = None

    if "Exportaciones" in pivot.columns and "Importaciones" in pivot.columns:
        pivot["balance_usd"] = pivot["Exportaciones"] - pivot["Importaciones"]
        pivot["perfil"] = pivot["balance_usd"].apply(
            lambda x: "🟢 Exportador neto" if x > 0
            else "🔴 Importador neto" if x < 0
            else "⚪ Balanceado"
        )
        pivot = pivot.sort_values("Exportaciones", ascending=False)

    return pivot

https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords


In [57]:
def datamx_eci_estados(year=2023):
    rows = datamx_get({
        "cube": "complexity_eci",
        "drilldowns": "Year,State",
        "measures": "ECI",
        "Year": year,
        "limit": "100",
        "locale": "es"
    })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows).rename(columns={
        "Year": "anio",
        "State": "estado",
        "State ID": "estado_id",
        "ECI": "eci"
    })

    df["fuente"] = "DataMexico"
    return df.sort_values("eci", ascending=False)

In [61]:
test_estado = datamx_get({
    "cube": "complexity_eci",
    "drilldowns": "Year,State",
    "measures": "ECI",
    "Year": 2024,
    "limit": "10",
    "locale": "es"
})
print(test_estado[:5])

URL base usada: https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords
[{'Year': 2024, 'State ID': 1, 'State': 'Aguascalientes', 'ECI': 0.5583919733762741}, {'Year': 2024, 'State ID': 2, 'State': 'Baja California', 'ECI': 0.9996783435344696}, {'Year': 2024, 'State ID': 3, 'State': 'Baja California Sur', 'ECI': 0.1049513965845108}, {'Year': 2024, 'State ID': 4, 'State': 'Campeche', 'ECI': -0.3435804694890976}, {'Year': 2024, 'State ID': 5, 'State': 'Coahuila de Zaragoza', 'ECI': 1.1709174513816833}]


In [62]:
test_dmx = datamx_get({
    "cube": "complexity_eci",
    "drilldowns": "Year",
    "measures": "ECI",
    "limit": "5",
    "locale": "es"
})
print(test_dmx[:2])

URL base usada: https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords
[{'Year': 2015, 'ECI': 1.4853059029620554e-10}, {'Year': 2016, 'ECI': -3.914486980595554e-10}]


In [65]:
def clasificar_eci(x):
    if x >= 1:
        return "Alto"
    elif x >= 0:
        return "Medio"
    else:
        return "Bajo"

df_eci = datamx_eci_estados(2024)
df_eci["nivel"] = df_eci["eci"].apply(clasificar_eci)

URL base usada: https://www.economia.gob.mx/apidatamexico/tesseract/data.jsonrecords


In [66]:
score = (
    df_denue
    .merge(df_eci, left_on="entidad_nombre", right_on="estado", how="left")
)

In [52]:
score

,id,clee,nombre,razon_social,actividad_economica,id_clase,id_sector,id_subsector,estrato,telefono,...,entidad_clave,entidad_nombre,fuente_x,fecha_extraccion,anio,estado_id,estado,eci,fuente_y,nivel
0,762690,09016541110003013001000000U0,ARELLANO AYALA ABOGADOS,NaN,Bufetes jurídicos,NaN,NaN,NaN,0 a 5 personas,NaN,...,9,Ciudad de México,DENUE_INEGI,2026-04-13,2023,9,Ciudad de México,0.874739,DataMexico,Medio
1,11647314,09011722513005291000086441M7,BARBACOA MALENO,NaN,Restaurantes con servicio de preparación de an...,NaN,NaN,NaN,0 a 5 personas,NaN,...,9,Ciudad de México,DENUE_INEGI,2026-04-13,2023,9,Ciudad de México,0.874739,DataMexico,Medio
2,6730131,09002468111000232001000000U6,BMW LINDAVISTA,LINDAVISTA MOTORS SA DE CV,Comercio al por menor de automóviles y camione...,NaN,NaN,NaN,101 a 250 personas,NaN,...,9,Ciudad de México,DENUE_INEGI,2026-04-13,2023,9,Ciudad de México,0.874739,DataMexico,Medio
3,6324838,09011431192000031001007670S7,CEDIS IZTAPALAPA DTS,COMERCIALIZADORA PEPSICO MEXICO S DE RL DE CV,Comercio al por mayor de botanas y frituras,NaN,NaN,NaN,31 a 50 personas,NaN,...,9,Ciudad de México,DENUE_INEGI,2026-04-13,2023,9,Ciudad de México,0.874739,DataMexico,Medio
4,6319760,09017465915000021000003945S8,FONDO NACIONAL PARA EL FOMENTO A LAS ARTESANIA...,FONDO NACIONAL PARA EL FOMENTO DE LAS ARTESANIAS,Comercio al por menor en tiendas de artesanías,NaN,NaN,NaN,11 a 30 personas,NaN,...,9,Ciudad de México,DENUE_INEGI,2026-04-13,2023,9,Ciudad de México,0.874739,DataMexico,Medio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1277473,10265316,15104713120001881000000000U9,ZYL GAMES,NaN,Casas de juegos electrónicos,NaN,NaN,NaN,0 a 5 personas,NaN,...,15,Estado de México,DENUE_INEGI,2026-04-13,2023,15,Estado de México,0.986706,DataMexico,Medio
1277474,9451346,15104466114000653000000000U5,ZYLKU IMPORTACIONES,ZYLKU IMPORTACIONES SA DE CV,"Comercio al por menor de cristalería, loza y u...",NaN,NaN,NaN,0 a 5 personas,NaN,...,15,Estado de México,DENUE_INEGI,2026-04-13,2023,15,Estado de México,0.986706,DataMexico,Medio
1277475,2533061,15076463310006681000000000U7,ZZENIA,NaN,Comercio al por menor de calzado,NaN,NaN,NaN,0 a 5 personas,7221812004.0,...,15,Estado de México,DENUE_INEGI,2026-04-13,2023,15,Estado de México,0.986706,DataMexico,Medio
1277476,11587588,15076463310014241000000000S0,ZZENIA,NaN,Comercio al por menor de calzado,NaN,NaN,NaN,0 a 5 personas,7221812004.0,...,15,Estado de México,DENUE_INEGI,2026-04-13,2023,15,Estado de México,0.986706,DataMexico,Medio


---
## 📊 Análisis DataMéxico

In [11]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Comercio exterior: exportaciones vs importaciones por estado (último año)
if not df_datamx.empty and 'estado' in df_datamx.columns:
    anio_max = df_datamx['anio'].max()
    df_plot = df_datamx[df_datamx['anio'] == anio_max].copy()

    pivot = df_plot.groupby(['estado','tipo_operacion'])['valor_usd'].sum().reset_index()
    fig = px.bar(
        pivot, x='estado', y='valor_usd', color='tipo_operacion',
        title=f'Exportaciones vs Importaciones por Estado — {anio_max}',
        labels={'valor_usd':'Valor USD','estado':'Estado','tipo_operacion':'Tipo'},
        barmode='group', color_discrete_map={'Exportaciones':'#2563EB','Importaciones':'#EF4444'},
        height=500
    )
    fig.update_xaxes(tickangle=45)
    fig.update_yaxes(tickformat='$,.0f')
    fig.show()

    # Top sectores exportadores
    top_sec = (
        df_plot[df_plot['tipo_operacion']=='Exportaciones']
        .groupby('sector_hs')['valor_usd'].sum()
        .nlargest(15).reset_index()
    ) if 'sector_hs' in df_plot.columns else pd.DataFrame()

    if not top_sec.empty:
        top_sec['sector_hs'] = top_sec['sector_hs'].str[:50]
        fig2 = px.bar(
            top_sec, x='valor_usd', y='sector_hs', orientation='h',
            title=f'Top 15 Sectores Exportadores — {anio_max}',
            color='valor_usd', color_continuous_scale='Blues',
            text='valor_usd', height=550
        )
        fig2.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
        fig2.update_layout(yaxis={'categoryorder':'total ascending'}, showlegend=False)
        fig2.update_xaxes(tickformat='$,.0f')
        fig2.show()
else:
    print('⚠️  Sin datos DataMéxico para graficar')


⚠️  Sin datos DataMéxico para graficar


In [ ]:
# Tendencia temporal de comercio exterior
if not df_datamx.empty:
    temporal = df_datamx.groupby(['anio','tipo_operacion'])['valor_usd'].sum().reset_index()
    fig = px.line(
        temporal, x='anio', y='valor_usd', color='tipo_operacion',
        title='Tendencia Comercio Exterior México',
        markers=True, height=400,
        color_discrete_map={'Exportaciones':'#2563EB','Importaciones':'#EF4444'},
        labels={'anio':'Año','valor_usd':'Valor USD','tipo_operacion':'Tipo'}
    )
    fig.update_yaxes(tickformat='$,.0f')
    fig.show()

    # Mapa de calor: sector x estado
    if 'sector_hs' in df_datamx.columns:
        heatmap_data = (
            df_datamx[df_datamx['tipo_operacion']=='Exportaciones']
            .groupby(['estado','sector_hs'])['valor_usd'].sum()
            .reset_index()
            .pivot(index='sector_hs', columns='estado', values='valor_usd')
            .fillna(0)
        )
        if not heatmap_data.empty:
            heatmap_data.index = heatmap_data.index.str[:40]
            fig3 = px.imshow(
                heatmap_data,
                title='Mapa de Calor — Exportaciones por Sector y Estado (USD)',
                color_continuous_scale='Blues',
                aspect='auto', height=600
            )
            fig3.show()


---
## 🔗 Cruce DataMéxico + Leads
> Identifica en qué estados y sectores exportadores están tus leads DENUE — señal de que son empresas con actividad comercial activa

In [ ]:
if not df_datamx.empty and 'df_denue' in globals() and not df_denue.empty:
    # Estados con más exportaciones
    top_estados_export = (
        df_datamx[df_datamx['tipo_operacion']=='Exportaciones']
        .groupby('estado')['valor_usd'].sum()
        .nlargest(10).index.tolist()
    ) if 'tipo_operacion' in df_datamx.columns else []

    print(f'🏆 Top estados exportadores: {top_estados_export}')

    # Leads DENUE en esos estados
    if top_estados_export and 'entidad_nombre' in df_denue.columns:
        leads_exportadores = df_denue[
            df_denue['entidad_nombre'].isin(top_estados_export)
        ]
        print(f'\n📋 Leads DENUE en estados exportadores: {len(leads_exportadores):,}')

        # Con contacto
        mask = (
            leads_exportadores['telefono'].str.len().gt(6).fillna(False) |
            leads_exportadores['email'].str.len().gt(5).fillna(False) |
            leads_exportadores['sitio_web'].str.len().gt(5).fillna(False)
        ) if all(c in leads_exportadores.columns for c in ['telefono','email','sitio_web']) else pd.Series(True, index=leads_exportadores.index)

        leads_exportadores_contacto = leads_exportadores[mask]
        print(f'   Con contacto: {len(leads_exportadores_contacto):,}')

        # Guardar como leads prioritarios
        f = os.path.join(RUTA_TRABAJO, f'leads_PRIORITARIOS_exportadores_{TIMESTAMP}.csv')
        leads_exportadores_contacto.to_csv(f, index=False, encoding='utf-8-sig')
        print(f'\n💾 {f}')

        # Visualizar distribución por estado
        dist = leads_exportadores_contacto['entidad_nombre'].value_counts().reset_index()
        dist.columns = ['Estado','Leads']
        fig = px.bar(
            dist, x='Estado', y='Leads',
            title='Leads DENUE con Contacto en Estados Exportadores',
            color='Leads', color_continuous_scale='Greens',
            text='Leads', height=400
        )
        fig.update_traces(texttemplate='%{text:,}', textposition='outside')
        fig.show()
    else:
        print('⚠️  Sin cruce posible: revisa columna entidad_nombre en df_denue')
else:
    print('⚠️  Necesitas df_datamx y df_denue para el cruce')


---
## 📊 Análisis BI
> Los datos ya están extraídos — ahora visualizamos, cruzamos y scorificamos

## 1️⃣ Carga de Datos
> Ajusta `RUTA_LEADS` si tus CSVs están en otra carpeta

In [ ]:
RUTA_LEADS = '.'   # ← cambia aquí si es necesario

# DENUE
archivos_denue = sorted(glob.glob(os.path.join(RUTA_LEADS, 'leads_denue_*.csv')))
if archivos_denue:
    df_denue = pd.concat([pd.read_csv(f, low_memory=False) for f in archivos_denue], ignore_index=True)
    df_denue = df_denue.drop_duplicates(subset=['id'])
    print(f'✅ DENUE: {len(df_denue):,} registros')
else:
    print('⚠️  No encontré leads_denue_*.csv'); df_denue = pd.DataFrame()

# CompraNet contratos
archivos_cnet = sorted(glob.glob(os.path.join(RUTA_LEADS, 'leads_compranet_*_contratos.csv')))
if archivos_cnet:
    df_cnet = pd.concat([pd.read_csv(f, low_memory=False) for f in archivos_cnet], ignore_index=True)
    print(f'✅ CompraNet contratos: {len(df_cnet):,} registros')
else:
    print('⚠️  No encontré leads_compranet_*_contratos.csv'); df_cnet = pd.DataFrame()

# CompraNet empresas
archivos_emp = sorted(glob.glob(os.path.join(RUTA_LEADS, 'leads_compranet_*_empresas.csv')))
if archivos_emp:
    df_emp = pd.concat([pd.read_csv(f, low_memory=False) for f in archivos_emp], ignore_index=True)
    df_emp = df_emp.drop_duplicates(subset=['rfc'])
    print(f'✅ CompraNet empresas: {len(df_emp):,} únicos por RFC')
else:
    print('⚠️  No encontré leads_compranet_*_empresas.csv'); df_emp = pd.DataFrame()

## 2️⃣ Resumen General KPIs

In [ ]:
resumen = []
if not df_denue.empty:
    resumen.append({
        'Fuente': '🏢 DENUE',
        'Registros': f"{len(df_denue):,}",
        'Con teléfono': f"{(df_denue['telefono'].str.len().gt(6)).sum():,}" if 'telefono' in df_denue.columns else 'N/A',
        'Con email':    f"{(df_denue['email'].str.len().gt(5)).sum():,}" if 'email' in df_denue.columns else 'N/A',
        'Con web':      f"{(df_denue['sitio_web'].str.len().gt(5)).sum():,}" if 'sitio_web' in df_denue.columns else 'N/A',
        'Entidades':    str(df_denue['entidad_nombre'].nunique()) if 'entidad_nombre' in df_denue.columns else 'N/A',
    })
if not df_emp.empty:
    monto = df_emp['monto_total_MXN'].sum() if 'monto_total_MXN' in df_emp.columns else 0
    resumen.append({
        'Fuente': '📋 CompraNet',
        'Registros': f"{len(df_emp):,}",
        'Con teléfono': 'N/A', 'Con email': 'N/A', 'Con web': 'N/A',
        'Monto total MXN': f'${monto:,.0f}',
    })
display(HTML(pd.DataFrame(resumen).to_html(index=False, border=0)))
print(f'\n📊 TOTAL LEADS: {len(df_denue)+len(df_emp):,}')

## 3️⃣ Distribución por Sector / Actividad Económica (DENUE)

In [ ]:
if not df_denue.empty and 'actividad_economica' in df_denue.columns:
    top_act = df_denue['actividad_economica'].value_counts().head(20).reset_index()
    top_act.columns = ['Actividad','Establecimientos']
    top_act['Actividad'] = top_act['Actividad'].str[:55]
    fig = px.bar(top_act, x='Establecimientos', y='Actividad', orientation='h',
                 title='Top 20 Actividades Económicas — DENUE',
                 color='Establecimientos', color_continuous_scale='Blues',
                 text='Establecimientos', height=600)
    fig.update_traces(texttemplate='%{text:,}', textposition='outside')
    fig.update_layout(yaxis={'categoryorder':'total ascending'}, showlegend=False)
    fig.show()

if not df_denue.empty and 'estrato' in df_denue.columns:
    estrato = df_denue['estrato'].value_counts().reset_index()
    estrato.columns = ['Estrato','Cantidad']
    fig2 = px.pie(estrato, names='Estrato', values='Cantidad',
                  title='Distribución por Tamaño de Empresa',
                  color_discrete_sequence=COLORS, hole=0.4)
    fig2.update_traces(textinfo='percent+label')
    fig2.show()

## 4️⃣ Tendencia Temporal — Contratos CompraNet

In [ ]:
if not df_cnet.empty and 'fecha_inicio' in df_cnet.columns:
    df_cnet['fecha_dt'] = pd.to_datetime(df_cnet['fecha_inicio'], errors='coerce', dayfirst=True)
    df_cnet['anio_mes'] = df_cnet['fecha_dt'].dt.to_period('M').astype(str)
    temporal = (
        df_cnet.groupby('anio_mes')
        .agg(contratos=('rfc','count'), monto=('monto_contrato','sum'))
        .reset_index().sort_values('anio_mes')
    )
    temporal = temporal[temporal['anio_mes'] >= '2022']
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(x=temporal['anio_mes'], y=temporal['contratos'],
                         name='# Contratos', marker_color='#2563EB', opacity=0.8), secondary_y=False)
    fig.add_trace(go.Scatter(x=temporal['anio_mes'], y=temporal['monto'],
                             name='Monto MXN', line=dict(color='#10B981', width=3), mode='lines+markers'), secondary_y=True)
    fig.update_layout(title='Contratos CompraNet por Mes', height=450,
                      legend=dict(orientation='h', y=1.1))
    fig.update_yaxes(title_text='# Contratos', secondary_y=False)
    fig.update_yaxes(title_text='Monto MXN', secondary_y=True, tickformat='$,.0f')
    fig.show()
else:
    print('⚠️  Sin datos temporales')

## 5️⃣ Top Leads por Score — CompraNet

In [ ]:
if not df_emp.empty and 'score_lead' in df_emp.columns:
    top50 = df_emp.nlargest(50,'score_lead').copy()
    if 'monto_total_MXN' in top50.columns:
        top50['monto_M'] = (top50['monto_total_MXN']/1_000_000).round(1)
    fig = px.scatter(
        top50,
        x='num_contratos' if 'num_contratos' in top50.columns else top50.index,
        y='monto_M' if 'monto_M' in top50.columns else 'score_lead',
        size='score_lead',
        color='tamanio_empresa' if 'tamanio_empresa' in top50.columns else None,
        hover_name='razon_social' if 'razon_social' in top50.columns else None,
        title='Top 50 Leads CompraNet — Score vs Contratos vs Monto',
        labels={'num_contratos':'# Contratos','monto_M':'Monto Total (M MXN)'},
        color_discrete_sequence=COLORS, height=550)
    fig.show()
    cols_show = [c for c in ['rfc','razon_social','num_contratos','monto_total_MXN','tamanio_empresa','score_lead'] if c in top50.columns]
    print('\n🏆 TOP 20 LEADS:')
    display(top50[cols_show].head(20).reset_index(drop=True))
else:
    print('⚠️  Sin datos empresas CompraNet')

## 6️⃣ Mapa Geográfico — DENUE

In [ ]:
if not df_denue.empty and 'latitud' in df_denue.columns:
    df_geo = df_denue.copy()
    df_geo['latitud']  = pd.to_numeric(df_geo['latitud'],  errors='coerce')
    df_geo['longitud'] = pd.to_numeric(df_geo['longitud'], errors='coerce')
    df_geo = df_geo.dropna(subset=['latitud','longitud'])
    df_geo = df_geo[(df_geo['latitud'].between(14,33)) & (df_geo['longitud'].between(-118,-86))]
    muestra = df_geo.sample(min(5000, len(df_geo)), random_state=42)
    mapa = folium.Map(location=[19.4,-99.1], zoom_start=6, tiles='CartoDB positron')
    cluster = MarkerCluster().add_to(mapa)
    for _, row in muestra.iterrows():
        popup_txt = f"<b>{str(row.get('nombre',''))[:40]}</b><br>{str(row.get('actividad_economica',''))[:50]}<br>📞 {row.get('telefono','')}"
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=4, color='#2563EB', fill=True, fill_opacity=0.7,
            popup=folium.Popup(popup_txt, max_width=250)
        ).add_to(cluster)
    mapa.save('mapa_leads_denue.html')
    print(f'✅ Mapa guardado: mapa_leads_denue.html ({len(muestra):,} puntos)')
    display(mapa)
else:
    print('⚠️  Sin coordenadas DENUE')

## 7️⃣ Cruce DENUE + CompraNet — Leads Premium

In [ ]:
df_cruce = pd.DataFrame()
if not df_denue.empty and not df_emp.empty:
    def limpiar(s):
        return str(s).upper().replace('SA DE CV','').replace('S.A. DE C.V.','').replace('SC','').replace('.','').strip()
    df_emp['_key']   = df_emp['razon_social'].apply(limpiar) if 'razon_social' in df_emp.columns else ''
    df_denue['_key'] = df_denue['nombre'].apply(limpiar)     if 'nombre'       in df_denue.columns else ''
    cruces = set(df_emp['_key']) & set(df_denue['_key']) - {''}
    print(f'🔗 Empresas en AMBAS fuentes: {len(cruces):,}')
    if cruces:
        contacto_cols = [c for c in ['telefono','email','sitio_web','colonia','ubicacion'] if c in df_denue.columns]
        df_cruce = df_emp[df_emp['_key'].isin(cruces)].merge(
            df_denue[df_denue['_key'].isin(cruces)][['_key']+contacto_cols].drop_duplicates('_key'),
            on='_key', how='left'
        ).drop(columns=['_key'])
        print(f'📋 Leads premium: {len(df_cruce):,}')
        cols = [c for c in ['razon_social','rfc','num_contratos','monto_total_MXN','score_lead','telefono','email','sitio_web','ubicacion'] if c in df_cruce.columns]
        display(df_cruce[cols].head(20).reset_index(drop=True))
        df_cruce[cols].to_csv('leads_PREMIUM_cruce.csv', index=False, encoding='utf-8-sig')
        print('💾 Guardado: leads_PREMIUM_cruce.csv')
else:
    print('⚠️  Necesitas ambas fuentes para el cruce')

## 8️⃣ Scoring Final — Posibles Clientes

In [ ]:
frames = []
if not df_emp.empty:
    d = df_emp.copy()
    d['fuente_principal'] = 'CompraNet'
    d['nombre_empresa']   = d.get('razon_social', pd.Series(dtype=str))
    d['score_base']       = d.get('score_lead', pd.Series(0.0, index=d.index))
    d['contratos_gov']    = d.get('num_contratos', 0)
    d['monto_MXN']        = d.get('monto_total_MXN', 0)
    d['tiene_contacto']   = False
    frames.append(d[['rfc','nombre_empresa','fuente_principal','score_base','contratos_gov','monto_MXN','tiene_contacto']])

if not df_denue.empty:
    d = df_denue.copy()
    d['fuente_principal'] = 'DENUE'
    d['nombre_empresa']   = d.get('nombre', pd.Series(dtype=str))
    d['rfc'] = ''; d['contratos_gov'] = 0; d['monto_MXN'] = 0
    t = d['telefono'].str.len().gt(6).fillna(False) if 'telefono'  in d.columns else pd.Series(False, index=d.index)
    m = d['email'].str.len().gt(5).fillna(False)    if 'email'     in d.columns else pd.Series(False, index=d.index)
    w = d['sitio_web'].str.len().gt(5).fillna(False)if 'sitio_web' in d.columns else pd.Series(False, index=d.index)
    d['score_base']     = t*30 + m*30 + w*20 + (t & m & w)*20
    d['tiene_contacto'] = t | m | w
    frames.append(d[['rfc','nombre_empresa','fuente_principal','score_base','contratos_gov','monto_MXN','tiene_contacto']])

if frames:
    df_score = pd.concat(frames, ignore_index=True)
    s_max = df_score['score_base'].max()
    df_score['score_100'] = ((df_score['score_base'] / s_max)*100).round(1) if s_max > 0 else 0
    df_score['clasificacion'] = df_score['score_100'].apply(
        lambda s: '🔥 HOT' if s >= 75 else ('🌡️  WARM' if s >= 40 else '❄️  COLD'))

    print('📊 DISTRIBUCIÓN DE LEADS:')
    for k,v in df_score['clasificacion'].value_counts().items():
        print(f'   {k}: {v:,} ({v/len(df_score)*100:.1f}%)')

    fig = px.histogram(df_score, x='score_100', color='fuente_principal',
                       nbins=50, title='Distribución de Score de Leads',
                       labels={'score_100':'Score (0-100)'},
                       color_discrete_sequence=['#2563EB','#10B981'],
                       barmode='overlay', opacity=0.75, height=400)
    fig.add_vline(x=75, line_dash='dash', line_color='red',    annotation_text='HOT')
    fig.add_vline(x=40, line_dash='dash', line_color='orange', annotation_text='WARM')
    fig.show()

    top_clientes = (
        df_score[df_score['tiene_contacto'] | (df_score['contratos_gov'] > 0)]
        .nlargest(30,'score_100').reset_index(drop=True)
    )
    print('\n🎯 TOP 30 POSIBLES CLIENTES:')
    display(top_clientes[['nombre_empresa','fuente_principal','score_100','clasificacion','contratos_gov','monto_MXN','tiene_contacto']])

    df_score.sort_values('score_100', ascending=False).to_csv('leads_MASTER_scored.csv', index=False, encoding='utf-8-sig')
    print(f'\n💾 leads_MASTER_scored.csv ({len(df_score):,} leads)')
else:
    print('⚠️  Sin datos para scoring')

## 9️⃣ Export a Excel — Un archivo, múltiples hojas

In [ ]:
archivo_excel = f"leads_BI_{pd.Timestamp.now().strftime('%Y%m%d')}.xlsx"
with pd.ExcelWriter(archivo_excel, engine='openpyxl') as writer:
    if 'df_score' in dir() and not df_score.empty:
        df_score.sort_values('score_100', ascending=False).to_excel(writer, sheet_name='MASTER_Leads', index=False)
    if not df_emp.empty:
        df_emp.to_excel(writer, sheet_name='CompraNet_Empresas', index=False)
    if not df_denue.empty:
        mask = (
            df_denue['telefono'].str.len().gt(6).fillna(False) |
            df_denue['email'].str.len().gt(5).fillna(False) |
            df_denue['sitio_web'].str.len().gt(5).fillna(False)
        ) if all(c in df_denue.columns for c in ['telefono','email','sitio_web']) else pd.Series(True, index=df_denue.index)
        df_denue[mask].to_excel(writer, sheet_name='DENUE_Con_Contacto', index=False)
    if not df_cruce.empty:
        df_cruce.to_excel(writer, sheet_name='Leads_PREMIUM_Cruce', index=False)
print(f'✅ Excel exportado: {archivo_excel}')